In [ ]:
# 任务描述：输入一块大矩阵，拆分为实际的矩阵乘法指令在硬件上调度执行
# 运算器尺寸可配置， MNK 三个参数 ，记运算器尺寸为 cM cN cK ,一周期可以执行 2 * cM * cN * cK 次浮点运算。
# 存储器尺寸参数与 MNK 尺寸严格绑定, 以(cM, cK) 为一个 ATile, (cN, cK) 为一个 BTile, (cM, cN) 为一个 CTile。
# 暂存器分为五组，分别是 A B As Bs C, A / B 暂存器中有 16 个可用项， C / D 暂存器有 32 个可用项，并分为两组用于 Pipeline。
# 暂存器由配置寄存器指定形状，记暂存器尺寸为 sM sN sK, 单位为 Tile。 A/As 尺寸为 (sM, sK)，B/Bs 尺寸为 (sN, sK), C 尺寸为 (sM, sN)
# 约束： sM * sN <= 32  ||  sM * sK <= 16  ||  sN * sK <= 16
#  (sM, sN, sK) [0, 1, 2, 3, 4] => [1, 2, 4, 8, 16]
# 循环方式，同样使用配置寄存器指定。以最内层循环到最外层循环的顺序，存在三种组合方式：
# 0: C Persistence == K-inner / 1: A Persistence == N-inner / 2: B Persistence == M-inner

# 子任务尺寸记为 (bM, bN, bK, lastM, lastN, lastK), 分别表示子任务块数，及最后一块的有效数据数量
# lastM = M % (sM * cM)
# lastN = N % (sN * cN)
# lastK = K % (sK * cK)
# bM = M / (sM * cM)
# bN = N / (sN * cN)
# bK = K / (sK * cK)

# 每个子任务尺寸，均小于 (sM, sN, sK), 会进一步拆分为 sM * sN * sK 条矩阵乘法指令及至多 (sM + sN) * sK + sM * sN 条加载指令。
# 每个子任务，对应元素级别的尺寸为 (last? lastM : sM * cMN, last? lastN : sN * cMN, last? lastK : sK * cK)

In [ ]:
# 子任务：块级别调度器
# 输入：计算尺寸 M N K, 暂存器形状配置 sM sN sK, Persistence Mode
# 输出：分块任务
def block_scheduler(M: int, N: int, K: int, sM: int, sN: int, sK: int, cM: int, cN: int, cK: int, mode: int):
    # 计算子任务尺寸
    cMN = max(cM, cN)
    lastM = M % (sM * cM)
    lastN = N % (sN * cN)
    lastK = K % (sK * cK)
    bM = M // (sM * cM)
    bN = N // (sN * cN)
    bK = K // (sK * cK)
    # 输出子任务序列
    # 确定 Persistence Dimension(循环最内层)
    loopDim = [bM, bN, bK]
    ldCmdDim = ['B', 'A', 'C']

    def listMoveRight(L: list, c: int):
        for i in range(c):
            L.insert(0, L.pop())
        return L

    loopDim = listMoveRight(loopDim, mode)
    ldCmdDim = listMoveRight(ldCmdDim, mode)

    cmdList = []
    
    PersistenceBank = 0
    InnerBank = 0
    for i0 in range(loopDim[0]):
        for i1 in range(loopDim[1]):
            cmdList.append('ld ' + ldCmdDim[2] + str(PersistenceBank))
            for i2 in range(loopDim[2]):
                cmdList.append('ld ' + ldCmdDim[1] + str(InnerBank))
                cmdList.append('ld ' + ldCmdDim[0] + str(InnerBank))
                cmdList.append('compute A{BankA}, B{BankB} => C{BankC}'.format(
                    BankC=str(PersistenceBank) if mode == 0 else str(InnerBank),
                    BankA=str(PersistenceBank) if mode == 1 else str(InnerBank),
                    BankB=str(PersistenceBank) if mode == 2 else str(InnerBank)
                    ))
                if mode != 0:
                    cmdList.append('st C' + str(InnerBank))
                InnerBank = 1 - InnerBank
            if mode == 0:
                cmdList.append('st C' + str(PersistenceBank))
            PersistenceBank = 1 - PersistenceBank
    
    return cmdList


In [12]:
l = block_scheduler(
    M = 64, N = 128, K = 256, sM = 4, sN = 4, sK = 4, cM = 4, cN = 8, cK = 32, mode = 2
)
print(l)

['ld B0', 'ld C0', 'ld A0', 'compute A0, B0 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B0 => C1', 'st C1', 'ld B1', 'ld C0', 'ld A0', 'compute A0, B1 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B1 => C1', 'st C1', 'ld B0', 'ld C0', 'ld A0', 'compute A0, B0 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B0 => C1', 'st C1', 'ld B1', 'ld C0', 'ld A0', 'compute A0, B1 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B1 => C1', 'st C1', 'ld B0', 'ld C0', 'ld A0', 'compute A0, B0 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B0 => C1', 'st C1', 'ld B1', 'ld C0', 'ld A0', 'compute A0, B1 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B1 => C1', 'st C1', 'ld B0', 'ld C0', 'ld A0', 'compute A0, B0 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B0 => C1', 'st C1', 'ld B1', 'ld C0', 'ld A0', 'compute A0, B1 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B1 => C1', 'st C1', 'ld B0', 'ld C0', 'ld A0', 'compute A0, B0 => C0', 'st C0', 'ld C1', 'ld A1', 'compute A1, B0 => C1', 'st C1', 